In [1]:
import sys
import pandas as pd
from pathlib import Path

from pydantic import BaseModel

import os
from dotenv import load_dotenv  
from pprint import pprint

from tqdm import tqdm
import constants

from prompting_utils import create_system_prompt
from model_utils import from_series_to_basemodel

import json

from mistralai import Mistral

from pathlib import Path

# Preliminari

In [ ]:
# Configurazione Mistral
client = Mistral(
    api_key=os.getenv("MISTRAL_API_KEY")
)

# Parametri
base_dir = Path.cwd().parent
SYSTEM_PROMPT_FILE_NAME = constants.SYSTEM_PROMPT_4
TEMPERATURE = 0.0

MODEL = constants.MISTRAL_LARGE_3
RESULTS_FILE_NAME = 'results_' + 'mistral_large_3' + '.jsonl'

PYDANTIC_MODEL = constants.RectalCancerStagingData

#Carica system prompt da file
system_prompt_path = base_dir / "data" / "prompts" / SYSTEM_PROMPT_FILE_NAME
system_prompt = create_system_prompt(system_prompt_path, PYDANTIC_MODEL)

In [3]:
if True:
    print(system_prompt)

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite RM.

Il tuo compito è estrarre informazioni strutturate dal referto RM fornito e restituire esclusivamente un oggetto JSON valido conforme allo schema seguente:

{"morfologia": "solido_polipoide | solido_anulare | mucinoso", "ore_inizio": "int | null", "ore_fine": "int | null", "spessore_parietale": "int | null", "estensione_cranio_caudale": "int | null", "distanza_oai": "int | null", "posizione": {"basso": "no | si", "medio": "no | si", "alto": "no | si", "giunzione": "no | si"}, "riflessione_peritoneale_anteriore": "sotto | cavallo | non_valutabile", "infiltrazione_tessuto_adiposo": "no | si_5mm | si_5mm_plus", "infiltrazione_sfinteri": "no | si", "infiltrazione_organi_extra": "no | si", "infiltrazione_organi_dettagli": {"pavimento_pelvico": "no | si", "altro": "no | si"}, "coinvolgimento_riflessione_peritoneale": "no | si", "coinvolgimento_fascia_mesorettale": "no | si", "numero_linfonodi_no

# Load Data

In [4]:
# Carichiamo i nostri file csv
file_names = {
    'validation': constants.VALIDATION_SPLIT_FILE_NAME,
    'test': constants.TEST_SPLIT_FILE_NAME,
}

paths = {split: Path('../data/' + file_name) for split, file_name in file_names.items()}

data = dict()
for split, path in paths.items():
    data[split] = pd.read_csv(path)

validation_data, test_data = data['validation'], data['test']

################################
# Convert float columns to Int64
################################
float_cols = test_data.select_dtypes("float").columns
for col in float_cols:
    test_data[col] = test_data[col].round().astype("Int64")
    validation_data[col] = validation_data[col].round().astype("Int64")
    
# Check duplicatest
assert set(test_data.id) & set(validation_data.id) == set(), "There are overlapping IDs between test and validation sets!"

print(f"{len(test_data) = }")
print(f"{len(validation_data) = }")

len(test_data) = 65
len(validation_data) = 66


# Generazione

## Preliminari generazione

In [5]:
# Funzione generatrice
def extract_data_from_report(model: str,
                             report_text: str,
                             system_prompt: str = system_prompt,
                             temperature: float = TEMPERATURE,
                             output_format: type[BaseModel] = PYDANTIC_MODEL):
    response = client.chat.parse(
        model=model,
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': report_text},
            
        ],
        response_format=output_format
    )
    return response

In [6]:
# Esempio
if True:
    result = extract_data_from_report(MODEL, data['test'].iloc[0]['report_text'])

In [7]:
if False: # To see the full output
    pprint(result.model_dump())
if False:  # To see the string output text
    print(result.choices[0].message.content)
if False:  # To see the parsed output as a pydantic BaseModel
    print(result.choices[0].message.parsed)
if False:
    print(result.output_parsed.model_dump(mode='json'))

## inference
Usiamo modelli non finetunati. Solo prompt engineering.

In [8]:
print(MODEL)
df = pd.concat([validation_data, test_data], ignore_index=True)
ids = []
actual = []
predicted = []
splits = []
for i in tqdm(range(df.shape[0])):
    row = df.iloc[i]
    splits.append(row['split'])
    output = extract_data_from_report(model=MODEL, report_text=row[constants.REPORT_COLUMN_NAME])
    real = from_series_to_basemodel(row, PYDANTIC_MODEL)
    ids.append(row[constants.ID_COMULM_NAME])
    actual.append(real.model_dump(mode='json'))
    if output is None:
        predicted.append('no output')
    else:
        predicted.append(output.choices[0].message.parsed.model_dump(mode='json'))

mistral-large-2512


100%|██████████| 131/131 [11:52<00:00,  5.44s/it]


In [9]:
results_dicts = []
assert len(ids) == len(actual) == len(predicted)
for id, act, pred, split in zip(ids, actual, predicted, splits):
    results_dicts.append(
        {
            'model': MODEL,
            'split': split,
            'id': int(id),
            'actual': act,
            'prediction': pred
        }
    )
# Salvataggio
with open(base_dir / 'data' / 'inference' / RESULTS_FILE_NAME, 'w', encoding='utf-8') as f:
    for r in results_dicts:
        f.write(json.dumps(r) + '\n')

In [10]:
output.model_dump()

{'id': 'fe9c1553e89c4392be5ae6efd43dd326',
 'object': 'chat.completion',
 'model': 'mistral-large-2512',
 'usage': {'prompt_tokens': 1338,
  'completion_tokens': 399,
  'total_tokens': 1737,
  'prompt_tokens_details': {'cached_tokens': 3040}},
 'created': 1770924539,
 'choices': [{'index': 0,
   'message': {'content': '{\n    "morfologia": "solido_anulare",\n    "ore_inizio": null,\n    "ore_fine": null,\n    "spessore_parietale": null,\n    "estensione_cranio_caudale": 40,\n    "distanza_oai": 50,\n    "posizione": {\n        "basso": "no",\n        "medio": "si",\n        "alto": "si",\n        "giunzione": "no"\n    },\n    "riflessione_peritoneale_anteriore": "cavallo",\n    "infiltrazione_tessuto_adiposo": "si_5mm_plus",\n    "infiltrazione_sfinteri": "no",\n    "infiltrazione_organi_extra": "no",\n    "infiltrazione_organi_dettagli": {\n        "pavimento_pelvico": "no",\n        "altro": "no"\n    },\n    "coinvolgimento_riflessione_peritoneale": "si",\n    "coinvolgimento_fasci